In [3]:
import numpy as np
from scipy.stats import norm
import pandas as pd

# ==========================================
# ALL INPUTS — CHANGE ANYTHING HERE
# ==========================================
# Grade Data
grades = ['RSS 1', 'RSS 3']
spot_prices = [760, 686.25]          # Latest market prices (LKR)
volatilities = [26.17, 23.49]     # Annualised volatility (%)
hist_means = [320.25, 307.88]     # Historical mean prices (LKR)

# Contract Parameters
r = 8.90                          # Risk-free rate 2025 January 10 (%)
T = 1.0                           # 1-year contract (as in your thesis)
prod_cost = 386.08                # Production cost floor (LKR)

# ==========================================
# BLACK-SCHOLES PUT OPTION FUNCTION
# ==========================================
def black_scholes_put(S, K, T, r, sigma):
    r_dec = r / 100
    sigma_dec = sigma / 100
    d1 = (np.log(S / K) + (r_dec + 0.5 * sigma_dec**2) * T) / (sigma_dec * np.sqrt(T))
    d2 = d1 - sigma_dec * np.sqrt(T)
    put_price = (K * np.exp(-r_dec * T) * norm.cdf(-d2)) - (S * norm.cdf(-d1))
    return put_price

# ==========================================
# RUN BLACK-SCHOLES FOR ALL GRADES AND TIERS
# ==========================================
results = []

for i in range(len(grades)):
    grade = grades[i]
    S = spot_prices[i]
    vol = volatilities[i]
    h_mean = hist_means[i]
    
    scenarios = {
        '100% Market'     : S,
        '95% Market'      : S * 0.95,
        'Historical Mean' : h_mean,
        'Production Cost' : prod_cost
    }
    
    for label, K in scenarios.items():
        premium_bs = black_scholes_put(S, K, T, r, vol)
        
        results.append({
            'Grade'                  : grade,
            'Scenario'               : label,
            'Strike (K)'             : round(K, 2),
            'BS Premium (LKR/kg)'    : round(premium_bs, 2)
        })

# ==========================================
# FINAL TABLE OUTPUT
# ==========================================
df_final = pd.DataFrame(results)
print("=== BLACK-SCHOLES PRICE-ONLY PREMIUMS (1-YEAR CONTRACT) ===")
print(df_final.to_string(index=False))

=== BLACK-SCHOLES PRICE-ONLY PREMIUMS (1-YEAR CONTRACT) ===
Grade        Scenario  Strike (K)  BS Premium (LKR/kg)
RSS 1     100% Market      760.00                47.70
RSS 1      95% Market      722.00                34.50
RSS 1 Historical Mean      320.25                 0.00
RSS 1 Production Cost      386.08                 0.07
RSS 3     100% Market      686.25                36.55
RSS 3      95% Market      651.94                25.37
RSS 3 Historical Mean      307.88                 0.00
RSS 3 Production Cost      386.08                 0.08
